In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.2 Optimizers

The optimizer turns gradients into weight updates. AdamW is the default: Adam's
per-parameter adaptive step (from running estimates of the gradient's first and
second moments) plus decoupled weight decay for regularization.

In [ ]:
from pocketlm import CharTokenizer
corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

from pocketlm import PocketLM, PocketLMConfig, init_weights, make_batch
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
g = torch.Generator().manual_seed(0)
first = None
steps = 60 if os.environ.get("POCKETLM_CI") else 300
for _ in range(steps):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    if first is None:
        first = loss.item()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("loss:", round(first, 3), "->", round(loss.item(), 3))

The adaptive moments let AdamW take confident steps on stable directions and
cautious ones on noisy parameters, which is why it just works across most models.

In [ ]:
assert loss.item() < first